In [1]:
# Dependencies for SB3 + Atari + training curves (matplotlib) + video (imageio)
%pip install -q stable-baselines3 gymnasium ale-py shimmy imageio pandas matplotlib
%pip install -q gymnasium[atari] autorom
!AutoROM --accept-license


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 6.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/

In [3]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
print("ALE/Pong-v5 available:", "ALE/Pong-v5" in gym.envs.registry)

ALE/Pong-v5 available: True


In [4]:
!mkdir -p /kaggle/working/Honorine-pong
!mkdir -p /kaggle/working/backups

In [5]:
%%writefile /kaggle/working/Honorine-pong/train.py
#!/usr/bin/env python3
"""
Train DQN on ALE/Pong-v5 with Stable-Baselines3.

Policy comparison (CNN vs MLP):
- `--policy cnn`: Convolutional features over screen input (standard for Atari / Pong).
- `--policy mlp`: Same DQN but `MlpPolicy` on flattened pixels — usually worse because spatial
  structure (ball/paddle location) is not exploited as effectively as with a CNN.
- `--policy both`: Train CNN and MLP in one run (two checkpoints under runs/<exp>/cnn|mlp).

Logging reward trends and episode length:
- During training, SB3 prints rollout `ep_rew_mean` and `ep_len_mean` in the console / TensorBoard.
- After each run, this script saves `runs/<exp>/<policy>/training_curves.png` from the Monitor CSV
  (per-episode reward + episode length, with a moving average).

Outputs:
- Save trained weights as `dqn_model.zip` (plus per-run paths in `hyperparameter_results.csv`).
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path
from typing import Dict, List

import ale_py
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage


def register_ale_envs():
    """Required on some Kaggle/Gymnasium setups so ALE/* ids are visible."""
    gym.register_envs(ale_py)


def build_env_fn(env_id: str, monitor_path: Path, flatten_obs: bool, render_mode: str | None = None):
    """Create one Atari env with consistent preprocessing."""

    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        try:
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
                grayscale_obs=True,
                scale_obs=False,
            )
        except TypeError:
            # Older SB3 AtariWrapper signature (Kaggle often has this)
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
            )

        if flatten_obs:
            env = gym.wrappers.FlattenObservation(env)
        env = Monitor(env, filename=str(monitor_path))
        return env

    return _make


def make_vec_env(env_id: str, policy_kind: str, monitor_path: Path, render_mode: str | None = None):
    flatten_obs = policy_kind.lower() == "mlp"
    vec_env = DummyVecEnv([build_env_fn(env_id, monitor_path, flatten_obs, render_mode=render_mode)])
    vec_env = VecMonitor(vec_env)
    if policy_kind.lower() == "cnn":
        # Stack 4 frames to provide temporal context for the CNN.
        vec_env = VecFrameStack(vec_env, n_stack=4)
        # SB3 CNN expects channel-first (C, H, W).
        vec_env = VecTransposeImage(vec_env)
    return vec_env


def load_monitor_dataframe(monitor_csv: Path) -> pd.DataFrame:
    if not monitor_csv.exists():
        return pd.DataFrame(columns=["r", "l", "t"])
    # First line in monitor.csv is metadata, second line is header.
    return pd.read_csv(monitor_csv, skiprows=1)


def save_training_plots(monitor_csv: Path, output_png: Path, policy_name: str):
    df = load_monitor_dataframe(monitor_csv)
    if df.empty:
        return

    rewards = df["r"].to_numpy()
    ep_len = df["l"].to_numpy()
    x = np.arange(1, len(df) + 1)

    window = min(25, len(df))
    reward_ma = pd.Series(rewards).rolling(window=window).mean()
    len_ma = pd.Series(ep_len).rolling(window=window).mean()

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(f"{policy_name} Training Trends")

    axes[0].plot(x, rewards, alpha=0.35, label="episode reward")
    axes[0].plot(x, reward_ma, label=f"moving avg ({window})")
    axes[0].set_ylabel("Reward")
    axes[0].legend()

    axes[1].plot(x, ep_len, alpha=0.35, label="episode length")
    axes[1].plot(x, len_ma, label=f"moving avg ({window})")
    axes[1].set_ylabel("Episode Length")
    axes[1].set_xlabel("Episode")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_png, dpi=140)
    plt.close(fig)


def evaluate_model(model: DQN, env_id: str, policy_kind: str, n_episodes: int = 5) -> Dict[str, float]:
    eval_monitor = Path("tmp_eval_monitor.csv")
    eval_env = make_vec_env(env_id, policy_kind=policy_kind, monitor_path=eval_monitor)
    episode_rewards: List[float] = []

    for _ in range(n_episodes):
        obs = eval_env.reset()
        done = False
        total_reward = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)  # Greedy (max-Q) evaluation
            obs, rewards, dones, infos = eval_env.step(action)
            total_reward += float(rewards[0])
            done = bool(dones[0])
        episode_rewards.append(total_reward)

    eval_env.close()
    if eval_monitor.exists():
        eval_monitor.unlink()

    return {
        "eval_mean_reward": float(np.mean(episode_rewards)),
        "eval_std_reward": float(np.std(episode_rewards)),
    }


def append_experiment_row(results_csv: Path, row: Dict[str, str | int | float]):
    results_csv.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_csv.exists()
    with results_csv.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def train_one_policy(args: argparse.Namespace, policy_kind: str, run_dir: Path) -> Dict[str, float]:
    run_dir.mkdir(parents=True, exist_ok=True)
    monitor_csv = run_dir / "monitor.csv"
    eval_monitor_csv = run_dir / "eval_monitor.csv"

    env = make_vec_env(args.env_id, policy_kind=policy_kind, monitor_path=monitor_csv)
    eval_env = make_vec_env(args.env_id, policy_kind=policy_kind, monitor_path=eval_monitor_csv)

    policy_name = "CnnPolicy" if policy_kind.lower() == "cnn" else "MlpPolicy"

    model = DQN(
        policy_name,
        env,
        learning_rate=args.lr,
        gamma=args.gamma,
        batch_size=args.batch_size,
        buffer_size=args.buffer_size,
        learning_starts=args.learning_starts,
        train_freq=args.train_freq,
        gradient_steps=args.gradient_steps,
        target_update_interval=args.target_update_interval,
        exploration_initial_eps=args.epsilon_start,
        exploration_final_eps=args.epsilon_end,
        exploration_fraction=args.epsilon_decay_fraction,
        tensorboard_log=str(args.tensorboard_dir),
        verbose=1,
        seed=args.seed,
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(run_dir / "best_model"),
        log_path=str(run_dir / "eval_logs"),
        eval_freq=args.eval_freq,
        n_eval_episodes=args.eval_episodes,
        deterministic=True,
        render=False,
    )

    model.learn(total_timesteps=args.total_timesteps, callback=eval_callback, progress_bar=True)

    final_model_path = run_dir / "dqn_model.zip"
    model.save(str(final_model_path))
    save_training_plots(monitor_csv, run_dir / "training_curves.png", policy_name=policy_name)

    metrics = evaluate_model(model, args.env_id, policy_kind=policy_kind, n_episodes=args.eval_episodes)
    metrics["policy"] = policy_kind
    metrics["final_model_path"] = str(final_model_path)

    with (run_dir / "metrics.json").open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    env.close()
    eval_env.close()
    return metrics


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train DQN for Atari Pong (SB3 + Gymnasium).")
    parser.add_argument("--env-id", type=str, default="ALE/Pong-v5")
    parser.add_argument("--policy", type=str, default="both", choices=["cnn", "mlp", "both"])
    parser.add_argument("--total-timesteps", type=int, default=1_000_000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--exp-name", type=str, default="exp01")
    parser.add_argument("--output-dir", type=Path, default=Path("runs"))
    parser.add_argument("--tensorboard-dir", type=Path, default=Path("tb_logs"))
    parser.add_argument("--eval-freq", type=int, default=50_000)
    parser.add_argument("--eval-episodes", type=int, default=5)

    # Hyperparameters requested by assignment
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--epsilon-start", type=float, default=1.0)
    parser.add_argument("--epsilon-end", type=float, default=0.05)
    parser.add_argument(
        "--epsilon-decay-fraction",
        type=float,
        default=0.1,
        help="Fraction of total timesteps over which epsilon decays.",
    )

    # Useful defaults for Atari DQN
    parser.add_argument("--buffer-size", type=int, default=100_000)
    parser.add_argument("--learning-starts", type=int, default=20_000)
    parser.add_argument("--train-freq", type=int, default=4)
    parser.add_argument("--gradient-steps", type=int, default=1)
    parser.add_argument("--target-update-interval", type=int, default=10_000)

    return parser.parse_args()


def main():
    register_ale_envs()
    args = parse_args()
    base_dir = args.output_dir / args.exp_name
    base_dir.mkdir(parents=True, exist_ok=True)

    policies = ["cnn", "mlp"] if args.policy == "both" else [args.policy]
    summary_rows: List[Dict[str, str | int | float]] = []

    for policy_kind in policies:
        run_dir = base_dir / policy_kind
        metrics = train_one_policy(args, policy_kind=policy_kind, run_dir=run_dir)
        summary_rows.append(
            {
                "member_name": "Honorine",
                "experiment_name": args.exp_name,
                "policy": policy_kind,
                "env_id": args.env_id,
                "timesteps": args.total_timesteps,
                "lr": args.lr,
                "gamma": args.gamma,
                "batch_size": args.batch_size,
                "epsilon_start": args.epsilon_start,
                "epsilon_end": args.epsilon_end,
                "epsilon_decay_fraction": args.epsilon_decay_fraction,
                "eval_mean_reward": round(float(metrics["eval_mean_reward"]), 3),
                "eval_std_reward": round(float(metrics["eval_std_reward"]), 3),
                "noted_behavior": "Fill after run (e.g., stable learning, high variance, over/under-exploration).",
                "model_path": str(run_dir / "dqn_model.zip"),
            }
        )

    # Save one top-level model file for assignment convenience.
    # Prefer CNN model by default (usually better on visual Atari input).
    preferred_policy = "cnn" if "cnn" in policies else policies[0]
    preferred_model = base_dir / preferred_policy / "dqn_model.zip"
    assignment_model_path = Path("dqn_model.zip")
    if preferred_model.exists():
        assignment_model_path.write_bytes(preferred_model.read_bytes())

    results_csv = args.output_dir / "hyperparameter_results.csv"
    for row in summary_rows:
        append_experiment_row(results_csv, row)

    print("\nTraining complete.")
    print(f"Assignment model: {assignment_model_path.resolve()}")
    print(f"Results table updated: {results_csv.resolve()}")
    print("Per-policy outputs saved in:", base_dir.resolve())


if __name__ == "__main__":
    main()

Writing /kaggle/working/Honorine-pong/train.py


In [6]:
%%writefile /kaggle/working/Honorine-pong/play.py
#!/usr/bin/env python3
"""
Load a trained DQN model and run greedy evaluation episodes.

GreedyQPolicy (evaluation):
- During training, DQN uses epsilon-greedy exploration.
- For evaluation, the assignment asks for GreedyQPolicy: always pick the action with highest Q-value.
- In Stable-Baselines3 this is done with: model.predict(obs, deterministic=True) each step.

visualization:
- `--render-mode human`: calls env.render() for a local GUI (best for live class demo).
- `--render-mode rgb_array` with `--save-video`: records frames from env.render() for Kaggle / headless.
"""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import List

import ale_py
import gymnasium as gym
import imageio.v2 as imageio
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage


def register_ale_envs():
    """Required on some Kaggle/Gymnasium setups so ALE/* ids are visible."""
    gym.register_envs(ale_py)


def build_env_fn(env_id: str, policy_kind: str, render_mode: str):
    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        try:
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
                grayscale_obs=True,
                scale_obs=False,
            )
        except TypeError:
            # Older SB3 AtariWrapper signature (Kaggle often has this)
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
            )

        if policy_kind.lower() == "mlp":
            env = gym.wrappers.FlattenObservation(env)
        return env

    return _make


def make_vec_env(env_id: str, policy_kind: str, render_mode: str):
    vec_env = DummyVecEnv([build_env_fn(env_id, policy_kind, render_mode)])
    if policy_kind.lower() == "cnn":
        vec_env = VecFrameStack(vec_env, n_stack=4)
        vec_env = VecTransposeImage(vec_env)
    return vec_env


def main():
    register_ale_envs()
    parser = argparse.ArgumentParser(description="Play Atari Pong with a trained DQN model.")
    parser.add_argument("--env-id", type=str, default="ALE/Pong-v5")
    parser.add_argument("--model-path", type=Path, default=Path("dqn_model.zip"))
    parser.add_argument("--policy", type=str, choices=["cnn", "mlp"], default="cnn")
    parser.add_argument("--episodes", type=int, default=3)
    parser.add_argument("--render-mode", type=str, default="human", choices=["human", "rgb_array"])
    parser.add_argument("--save-video", action="store_true", help="Save video (useful for Kaggle).")
    parser.add_argument("--video-path", type=Path, default=Path("pong_agent.mp4"))
    args = parser.parse_args()

    if not args.model_path.exists():
        raise FileNotFoundError(f"Model not found: {args.model_path}")

    env = make_vec_env(args.env_id, args.policy, args.render_mode)
    model = DQN.load(str(args.model_path), env=env)

    all_returns: List[float] = []
    frames: List[np.ndarray] = []

    for episode in range(args.episodes):
        obs = env.reset()
        done = False
        ep_return = 0.0

        while not done:
            # deterministic=True = greedy action (max Q-value)
            action, _state = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = env.step(action)
            ep_return += float(rewards[0])
            done = bool(dones[0])

            if args.render_mode == "rgb_array" and args.save_video:
                frame = env.render()
                if isinstance(frame, np.ndarray):
                    frames.append(frame)

        all_returns.append(ep_return)
        print(f"Episode {episode + 1}/{args.episodes} return: {ep_return:.2f}")

    print("\nEvaluation done.")
    print(f"Mean return over {args.episodes} episodes: {np.mean(all_returns):.2f}")
    print(f"Std return: {np.std(all_returns):.2f}")

    if args.save_video and frames:
        imageio.mimsave(args.video_path, frames, fps=30)
        print(f"Saved video: {args.video_path.resolve()}")

    env.close()


if __name__ == "__main__":
    main()

Writing /kaggle/working/Honorine-pong/play.py


In [7]:
# Cell 5: Verify files exist
!ls -la /kaggle/working/Honorine-pong

total 28
drwxr-xr-x 2 root root  4096 Mar 19 18:00 .
drwxr-xr-x 5 root root  4096 Mar 19 18:00 ..
-rw-r--r-- 1 root root  4187 Mar 19 18:00 play.py
-rw-r--r-- 1 root root 11237 Mar 19 18:00 train.py


In [ ]:
# Train
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp01 \
  --total-timesteps 1000000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 21:08:22.897324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773954502.919710     385 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773954502.926248     385 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773954502.944012     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773954502.944040     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773954502.944044     385 computation_placer.cc:177] computation placer alr

In [8]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp01 --total-timesteps 150000 --lr 1e-4 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 18:01:17.797331: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773943277.974895     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773943278.025103     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773943278.444674     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773943278.444730     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773943278.444734     117 computation_placer.cc:177] computation placer alr

In [9]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp02 --total-timesteps 150000 --lr 5e-5 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 18:14:13.291603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773944053.312972     142 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773944053.319461     142 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773944053.337059     142 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773944053.337085     142 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773944053.337088     142 computation_placer.cc:177] computation placer alr

In [10]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy mlp --exp-name exp03 --total-timesteps 100000 --lr 1e-4 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 30000 --eval-episodes 3

2026-03-19 18:28:38.360146: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773944918.382632     167 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773944918.389203     167 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773944918.406186     167 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773944918.406218     167 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773944918.406222     167 computation_placer.cc:177] computation placer alr

In [21]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp04 --total-timesteps 150000 --lr 2e-4 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

----------------------------------━━━━━━━━━━━━━━━━━━━━ 28,774/150,000  [ 0:02:12 < 0:13:53 , 146 it/s ]
| rollout/            |          |
|    ep_len_mean      | 203      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 140      |
|    fps              | 216      |
|    time_elapsed     | 133      |
|    total_timesteps  | 28781    |
| train/              |          |
|    learning_rate    | 0.0002   |
|    loss             | 0.0242   |
|    n_updates        | 2195     |
----------------------------------
----------------------------------0m╺━━━━━━━━━━━━━━━━━ 43,603/150,000  [ 0:03:51 < 0:11:27 , 155 it/s ]
| rollout/            |          |
|    ep_len_mean      | 195      |
|    ep_rew_mean      | -20.9    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 216      |
|    fps              | 188      |
|    time_elapsed     | 231      |
|    total_timesteps  

In [22]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp05 --total-timesteps 150000 --lr 1e-4 --gamma 0.95 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 16:17:07.790881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773937027.815160     200 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773937027.822671     200 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773937027.842555     200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773937027.842596     200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773937027.842602     200 computation_placer.cc:177] computation placer alr

In [11]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp06 --total-timesteps 150000 --lr 1e-4 --gamma 0.999 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 18:38:18.647334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773945498.669938     190 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773945498.676754     190 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773945498.694325     190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773945498.694376     190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773945498.694380     190 computation_placer.cc:177] computation placer alr

In [17]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp07 --total-timesteps 150000 --lr 1e-4 --gamma 0.99 --batch-size 64 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 19:08:15.235821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773947295.258279     292 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773947295.264784     292 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773947295.281277     292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773947295.281308     292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773947295.281312     292 computation_placer.cc:177] computation placer alr

In [18]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp08 --total-timesteps 150000 --lr 1e-4 --gamma 0.99 --batch-size 128 --epsilon-start 1.0 --epsilon-end 0.10 --epsilon-decay-fraction 0.10 --buffer-size 50000 --eval-episodes 3

2026-03-19 19:21:13.061706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773948073.083911     315 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773948073.090445     315 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773948073.107234     315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773948073.107262     315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773948073.107267     315 computation_placer.cc:177] computation placer alr

In [19]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp09 --total-timesteps 300000 --lr 1e-4 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.01 --epsilon-decay-fraction 0.20 --buffer-size 50000 --eval-episodes 3

2026-03-19 19:37:02.760518: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773949022.787564     338 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773949022.797778     338 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773949022.818356     338 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773949022.818393     338 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773949022.818400     338 computation_placer.cc:177] computation placer alr

In [20]:
!python /kaggle/working/Honorine-pong/train.py --env-id ALE/Pong-v5 --policy cnn --exp-name exp10 --total-timesteps 500000 --lr 1e-4 --gamma 0.99 --batch-size 32 --epsilon-start 1.0 --epsilon-end 0.05 --epsilon-decay-fraction 0.05 --buffer-size 50000 --eval-episodes 3

2026-03-19 20:12:21.755765: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773951141.778647     361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773951141.785231     361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773951141.802886     361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773951141.802916     361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773951141.802920     361 computation_placer.cc:177] computation placer alr

In [19]:
# --- FIXED Honorine Script: No more "No space left" errors ---
from pathlib import Path
import os
import pandas as pd

ROOT = Path("/kaggle/working")
RUNS = ROOT / "runs"
CSV_PATH = RUNS / "hyperparameter_results.csv"

print("=" * 60)
print("KAGGLE WORKING — EXPERIMENT FILES")
print("=" * 60)

# 1) Load hyperparameter table (This is safe, it won't delete your data!)
if not CSV_PATH.exists():
    print(f"MISSING: {CSV_PATH}")
else:
    df = pd.read_csv(CSV_PATH)
    df_sorted = df.sort_values("eval_mean_reward", ascending=False).reset_index(drop=True)
    print("ALL EXPERIMENTS (best reward first):\n")
    
    # This displays your table perfectly for the rubric
    show = [c for c in ["experiment_name", "policy", "timesteps", "lr", "gamma", "batch_size", 
                        "epsilon_start", "epsilon_end", "eval_mean_reward"] if c in df_sorted.columns]
    display(df_sorted[show])

    best = df_sorted.iloc[0]
    best_path = Path(str(best["model_path"]))
    
    print("\nBEST MODEL LOCATED:")
    print(f"  Experiment: {best.get('experiment_name', '?')}")
    print(f"  Reward:     {best['eval_mean_reward']}")
    print(f"  Location:   {best_path}")

    # 2) WE REMOVED SHUTIL.COPY2 HERE TO PREVENT "NO SPACE LEFT ON DEVICE"
    # We will just point our play script directly to the 'best_path' instead.

print("\n" + "=" * 60)

KAGGLE WORKING — EXPERIMENT FILES
ALL EXPERIMENTS (best reward first):



,experiment_name,policy,timesteps,lr,gamma,batch_size,epsilon_start,epsilon_end,eval_mean_reward
0,exp01,cnn,1000000,0.00010,0.990,32,1.0,0.05,-5.800
1,exp10,cnn,500000,0.00010,0.990,32,1.0,0.05,-6.667
2,exp08,cnn,150000,0.00010,0.990,128,1.0,0.10,-15.333
3,exp09,cnn,300000,0.00010,0.990,32,1.0,0.01,-16.000
4,exp01,cnn,150000,0.00010,0.990,32,1.0,0.05,-17.000
5,exp07,cnn,150000,0.00010,0.990,64,1.0,0.05,-17.333
6,exp06,cnn,150000,0.00010,0.999,32,1.0,0.05,-17.333
7,exp07,cnn,150000,0.00010,0.990,64,1.0,0.05,-18.000
8,exp02,cnn,150000,0.00005,0.990,32,1.0,0.05,-18.333
9,exp03,mlp,100000,0.00010,0.990,32,1.0,0.05,-21.000



BEST MODEL LOCATED:
  Experiment: exp01
  Reward:     -5.8
  Location:   runs/exp01/cnn/dqn_model.zip



In [18]:
# Set CUDA_VISIBLE_DEVICES to empty to force the script to use the CPU
# This avoids the "no kernel image" error caused by the Tesla P100 GPU mismatch
!export CUDA_VISIBLE_DEVICES="" && python /kaggle/working/Honorine-pong/play.py \
  --env-id ALE/Pong-v5 \
  --model-path /kaggle/working/dqn_model.zip \
  --policy cnn \
  --episodes 3 \
  --render-mode rgb_array \
  --save-video \
  --video-path /kaggle/working/pong_agent.mp4

2026-03-21 01:17:58.233363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774055878.256008     194 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774055878.263846     194 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774055878.284105     194 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774055878.284137     194 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774055878.284140     194 computation_placer.cc:177] computation placer alr